# Pump It Up: Predicting Waterpoint Operating Condition

AI7101 — Machine Learning with Python  
Majid Alredha · Hamad Alhosani · Hamdan Alhammadi

This project classifies Tanzanian waterpoints as **functional**, **functional needs repair**, or **non-functional**. It compares eight classifiers using macro-F1, selects a model and repair decision threshold, and evaluates an illustrative inspection policy.

The selected **XGBoost model with a repair threshold of 0.27** achieves holdout accuracy of **81.18%** and macro-F1 of **0.7144**. The analysis addresses recorded operating condition; prospective evaluation would be needed to assess future failure or operational benefits.


## 1. Setup and workflow

The workflow reserves a holdout set, explores the development data, prepares features, compares classifiers, selects a model and prediction rule, evaluates the holdout, and generates competition predictions.

Full mode (`QUICK_RUN=False`) uses **53,460 development records**, five-fold cross-validation, and a **5,940-record holdout**. Quick mode uses a stratified sample of 9,000 development records and three folds, and omits holdout evaluation and submission generation.

All classifiers use the same search records and validation folds. The random seed is **42**, package versions are printed below, and exported results are written to `outputs/`.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import xgboost
from IPython.display import display
from joblib import Memory, parallel_backend
from joblib import hash as joblib_hash
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import (
    BaggingClassifier,
    ExtraTreesClassifier,
    RandomForestClassifier,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.utils.validation import check_is_fitted
from xgboost import XGBClassifier


In [ ]:
SEED = 42
QUICK_RUN = False
N_SPLITS = 3 if QUICK_RUN else 5
PRIMARY_METRIC = "macro_f1"
RUN_FINAL_EVALUATION = True
BATCH_SIZE = 500

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data" / "raw").is_dir():
    REPO_ROOT = REPO_ROOT.parent
DATA_DIR = REPO_ROOT / "data" / "raw"
if not DATA_DIR.is_dir():
    raise FileNotFoundError("Run from the repository root or notebooks/.")
OUTPUT_DIR = REPO_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

CLASS_NAMES = ["functional", "functional needs repair", "non functional"]
label_to_number = {name: number for number, name in enumerate(CLASS_NAMES)}
number_to_label = {number: name for name, number in label_to_number.items()}

sns.set_theme(style="whitegrid")
sklearn.set_config(working_memory=128)

print("scikit-learn:", sklearn.__version__)
print("XGBoost:", xgboost.__version__)
print("Quick run:", QUICK_RUN)
print("CV folds:", N_SPLITS)


def invalidate_selection(invalidate_search=True):
    """Reset downstream results when data or model selection changes."""
    global search_ready, holdout_evaluated, evaluated_signature
    if invalidate_search:
        search_ready = False
    holdout_evaluated = False
    evaluated_signature = None
    for name in ["winner", "final_model", "holdout_pred"]:
        globals().pop(name, None)


def current_selection_signature():
    return (id(winner), joblib_hash(winner.get_params(deep=True)))


def require_current_holdout():
    if not holdout_evaluated or evaluated_signature != current_selection_signature():
        raise RuntimeError("Evaluate the selected model and decision rule first.")


invalidate_selection()
# Cache fitted preprocessing in a temporary directory for this session.
previous_cache = globals().get("preprocess_cache_dir")
if previous_cache is not None:
    previous_cache.cleanup()
preprocess_cache_dir = TemporaryDirectory(prefix="pump-preprocessing-")
pipeline_memory = Memory(preprocess_cache_dir.name, verbose=0)


## 2. Load and align the data

The dataset contains **59,400 labeled waterpoints**, **39 predictor fields**, and **14,850 unlabeled competition records**. Predictors describe location, infrastructure, water availability, funding, and management. Labels are joined to predictors by the unique waterpoint `id`. Alignment checks verify unique IDs, complete labels, disjoint training and competition IDs, and matching predictor columns.

The identifier is excluded from model inputs. The target is encoded as follows:

| Label | Operating condition | Training records | Share |
|---|---|---:|---:|
| `0` | Functional | 32,259 | 54.31% |
| `1` | Functional needs repair | 4,317 | 7.27% |
| `2` | Non-functional | 22,824 | 38.42% |

Competition records are used only for final prediction. Field and target definitions follow the [DrivenData dataset documentation](https://www.drivendata.org/competitions/7/pump-it-up-data-mining-the-water-table/page/25/).


In [ ]:
train_values = pd.read_csv(DATA_DIR / "train_values.csv")
train_labels = pd.read_csv(DATA_DIR / "train_labels.csv")
test_values = pd.read_csv(DATA_DIR / "test_values.csv")
submission_format = pd.read_csv(DATA_DIR / "submission_format.csv")

# Verify unique IDs, aligned labels, disjoint train/test records, and matching fields.
for frame in [train_values, train_labels, test_values, submission_format]:
    assert frame["id"].is_unique

assert set(train_values["id"]) == set(train_labels["id"])
assert set(test_values["id"]) == set(submission_format["id"])
assert set(train_values["id"]).isdisjoint(test_values["id"])
assert set(train_values.columns) == set(test_values.columns)

df = train_values.merge(train_labels, on="id", how="left", validate="one_to_one")

assert df["status_group"].notna().all()
assert set(df["status_group"]) == set(CLASS_NAMES)

# Exclude the target and identifier from model inputs.
X = df.drop(columns=["id", "status_group"])

# 0 = functional; 1 = functional needs repair; 2 = non-functional.
y = df["status_group"].map(label_to_number).astype(int)

X_test = test_values.drop(columns="id")

print("Training inputs:", X.shape)
print("Training target:", y.shape)
print("Competition test inputs:", X_test.shape)

display(df.head())


## 3. Reserve a 10% holdout set

A seeded, stratified split assigns **53,460 records (90%)** to development and **5,940 records (10%)** to the holdout. Exploration, hyperparameter selection, and threshold selection use development records.

Each development fold contains **42,768 training records** and **10,692 validation records**. Every development record appears in one validation fold. Threshold selection uses the first fold's validation records, with each shortlisted configuration fitted on that fold's training records.

The holdout is excluded from model fitting and selection, providing a separate evaluation of the chosen model and decision rule. Section 10 discusses the limits of this evaluation.


In [ ]:
invalidate_selection()

# Reserve a stratified holdout before exploration and model selection.

X_dev, X_holdout, y_dev, y_holdout = train_test_split(
    X, y, test_size=0.10, stratify=y, random_state=SEED
)

eda = X_dev.copy()
eda["status_group"] = y_dev.map(number_to_label)

print("Development rows:", len(X_dev))
print("Holdout rows:", len(X_holdout))

class_summary = eda["status_group"].value_counts().to_frame("count")
class_summary["percent"] = 100 * class_summary["count"] / len(eda)

display(class_summary)


## 4. Explore the development data

The development set contains **29,033 functional waterpoints (54.31%)**, **3,885 repair cases (7.27%)**, and **20,542 non-functional waterpoints (38.42%)**. A majority-class predictor achieves about 54% accuracy while detecting no repair cases, motivating evaluation beyond accuracy.

`scheme_name` is missing in **48.50%** of development records. Waterpoint names, subvillages, and wards also have many distinct values, making sparsely represented categories difficult to estimate reliably.

Literal missingness does not capture every data-quality issue. Nonpositive construction years and invalid coordinate pairs require validity rules before imputation.


In [ ]:
# Show class imbalance and literal missingness before applying validity rules.

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(
    data=eda, y="status_group", order=CLASS_NAMES, color="steelblue", ax=axes[0]
)
axes[0].set(title="Development class counts", xlabel="Pumps", ylabel="")

missing = X_dev.isna().mean().mul(100).sort_values(ascending=False)
missing = missing[missing > 0]

sns.barplot(x=missing.values, y=missing.index, color="coral", ax=axes[1])
axes[1].set(title="Missing values", xlabel="Percent missing", ylabel="")

plt.tight_layout()
plt.show()


In [ ]:
display(
    pd.DataFrame(
        {"unique_values": X_dev.nunique(), "missing_percent": X_dev.isna().mean() * 100}
    )
)


In [ ]:
# Compare within-category proportions alongside category counts.


def plot_status_by_category(feature):
    proportions = pd.crosstab(
        eda[feature], eda["status_group"], normalize="index"
    ).reindex(columns=CLASS_NAMES, fill_value=0)

    plt.figure(figsize=(10, 4))

    sns.heatmap(proportions, annot=True, fmt=".1%", cmap="Blues", vmin=0, vmax=1)

    plt.title(f"Pump status within each {feature}")
    plt.xlabel("Pump status")
    plt.tight_layout()
    plt.show()


plot_status_by_category("quantity")


In [ ]:
plot_status_by_category("waterpoint_type")


In [ ]:
for feature in ["quantity", "waterpoint_type"]:
    display(eda[feature].value_counts().to_frame("pumps"))


### Interpretation of the exploratory analysis

Water availability is strongly associated with operating condition: **96.8% of dry waterpoints** are non-functional. Waterpoint type also separates groups; **82.4% of the `other` category** is non-functional. These patterns support retaining both predictors and considering nonlinear models.

Category counts qualify these percentages. Only **six development records** have waterpoint type `dam`, so their proportions offer little evidence about future cases. Geographic clustering motivates validation across regions as well as random splits. These descriptive associations do not establish causation.


## 5. Prepare features

Deterministic rules applied to each row produce **37 input fields before encoding** for development, holdout, and competition data. These transformations use neither target labels nor statistics from other records.

Imputation, rare-category grouping, encoding, and optional scaling are learned within model pipelines. During cross-validation, these transformations are fitted only on each fold's training records.


### Feature-engineering decisions

| Transformation | Rationale |
|---|---|
| Drop `recorded_by` and `quantity_group` | The first is constant in development data; the second exactly duplicates `quantity`. |
| Drop `wpt_name`, `subvillage`, and raw `scheme_name` | Reduce sparsely supported categories; retain scheme-name availability through `scheme_name_missing`. |
| Replace `ward` with `ward_id` combining LGA and ward | Distinguish repeated ward names while retaining local information. |
| Retain `source_class` and `extraction_type_class` | Provide broader representations alongside detailed source and extraction fields; remove other overlapping grouped variants. |
| Validate construction year; derive pump age and recording year | Treat nonpositive years and construction after recording as missing; represent age and collection period. |
| Encode recording month with sine and cosine | Represent seasonality while keeping December and January adjacent. |
| Flag and replace invalid coordinate pairs | Zero longitude, latitude within 0.000001° of zero, or a missing coordinate marks the pair as missing. |
| Add coordinate-grid IDs at 0.05° and 0.10° resolution | Represent local context alongside continuous coordinates; invalid pairs remain missing. |
| Add zero indicators for population, elevation, and `amount_tsh` | Preserve potentially informative zeros without treating every zero as missing. |
| Replace population and `amount_tsh` with `log1p` versions | Reduce skew after clipping negative values to zero. `amount_tsh` denotes total static head in the dataset documentation. |
| Combine region and district codes into `district_id` | Represent administrative codes as categories rather than numeric distances. |
| Normalize categorical text | Standardize case and surrounding whitespace; treat empty strings as missing. |

The following original predictors remain alongside the engineered features:

| Information | Retained fields |
|---|---|
| Location and elevation | `longitude`, `latitude`, `gps_height`, `basin`, `region`, `lga` |
| Funding and installation | `funder`, `installer` |
| Administration and community | `num_private`, `public_meeting`, `scheme_management`, `management`, `permit`, `payment` |
| Infrastructure and water | `construction_year`, `extraction_type`, `extraction_type_class`, `water_quality`, `quantity`, `source`, `source_class`, `waterpoint_type` |

Recording date, raw ward, region and district codes, population, and `amount_tsh` are replaced by their engineered representations. Coordinate grids use angular units rather than fixed physical distances.


In [ ]:
invalidate_selection()

# Apply deterministic row-level rules; learn imputation and encoding in each fold.

DROP_COLUMNS = [
    "recorded_by",  # Constant in these data.
    "wpt_name",  # Many nearly unique names.
    "subvillage",  # High cardinality.
    "scheme_name",  # High cardinality and substantial missingness.
    "ward",  # Replaced by a ward identifier qualified by LGA.
    "quantity_group",  # Duplicate of quantity.
    "extraction_type_group",
    "management_group",
    "payment_type",
    "quality_group",
    "source_type",
    "waterpoint_type_group",
]

# Confirm the constant field and exact duplicate using development records.
assert X_dev["recorded_by"].nunique(dropna=False) == 1
assert X_dev["quantity"].equals(X_dev["quantity_group"])


def engineer_features(frame):
    data = frame.copy()

    # Normalize categories before constructing location and missingness features.
    categorical = data.select_dtypes(include=["object", "string", "bool"]).columns
    for column in categorical:
        data[column] = (
            data[column]
            .map(
                lambda value: str(value).strip().lower() if pd.notna(value) else np.nan
            )
            .replace("", np.nan)
        )

    data["ward_id"] = data["lga"].str.cat(data["ward"], sep="::")
    data["scheme_name_missing"] = data["scheme_name"].isna().astype(int)

    # Year 0 is invalid; construction after observation is inconsistent.
    recorded = pd.to_datetime(data["date_recorded"], errors="coerce")
    year = pd.to_numeric(data["construction_year"], errors="coerce")
    year = year.mask((year <= 0) | (year > recorded.dt.year))

    data["construction_year"] = year
    data["pump_age"] = recorded.dt.year - year
    data["record_year"] = recorded.dt.year

    # Cyclic month features place December near January.
    month = recorded.dt.month
    data["month_sin"] = np.sin(2 * np.pi * (month - 1) / 12)
    data["month_cos"] = np.cos(2 * np.pi * (month - 1) / 12)

    # Replace zero-longitude, near-zero-latitude, or incomplete coordinate pairs.
    bad_coordinates = (
        data["longitude"].eq(0)
        | np.isclose(data["latitude"], 0, atol=1e-6)
        | data[["longitude", "latitude"]].isna().any(axis=1)
    )

    data["coordinates_missing"] = bad_coordinates.astype(int)
    data.loc[bad_coordinates, ["longitude", "latitude"]] = np.nan

    # Geographic cells capture local context at two angular resolutions.
    for name, cell_size in [("geo_grid_005", 0.05), ("geo_grid_010", 0.10)]:
        longitude_cell = np.floor(data["longitude"] / cell_size).astype("Int64")
        latitude_cell = np.floor(data["latitude"] / cell_size).astype("Int64")
        data[name] = (
            longitude_cell.astype(str) + "::" + latitude_cell.astype(str)
        ).mask(bad_coordinates)

    # Preserve ambiguous zeros and give models explicit zero indicators.
    for column in ["population", "gps_height", "amount_tsh"]:
        data[column + "_is_zero"] = data[column].eq(0).astype(int)

    # Clip negative values and log-transform population and static head.
    for column in ["population", "amount_tsh"]:
        data[column + "_log1p"] = np.log1p(data[column].clip(lower=0))

    # Administrative codes are category labels, not numeric distances.
    data["district_id"] = (
        data["region_code"].astype(str) + "_" + data["district_code"].astype(str)
    )

    data = data.drop(
        columns=DROP_COLUMNS
        + ["date_recorded", "region_code", "district_code", "population", "amount_tsh"]
    )

    return data


X_dev_clean = engineer_features(X_dev)
X_holdout_clean = engineer_features(X_holdout)
X_test_clean = engineer_features(X_test)

display(X_dev_clean.head())
print("Prepared development shape:", X_dev_clean.shape)


In [ ]:
# Explore pump age and spatial patterns in the development data.

plot_data = X_dev_clean.copy()
plot_data["status_group"] = y_dev.map(number_to_label)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.boxplot(
    data=plot_data,
    x="pump_age",
    y="status_group",
    order=CLASS_NAMES,
    showfliers=False,
    color="lightblue",
    ax=axes[0],
)
axes[0].set(title="Age at recording", ylabel="")

sample = plot_data.sample(n=min(5000, len(plot_data)), random_state=SEED)

sns.scatterplot(
    data=sample,
    x="longitude",
    y="latitude",
    hue="status_group",
    hue_order=CLASS_NAMES,
    alpha=0.4,
    s=12,
    ax=axes[1],
)
axes[1].set_title("Location and pump status")

plt.tight_layout()
plt.show()


## 6. Compare models with cross-validation

**Macro-F1 is the primary selection metric.** It averages the three class-specific F1 scores equally, giving the rare repair class the same contribution as each larger class. Each class's F1 balances precision and recall. Accuracy and repair recall are also reported.

Every classifier uses the same shuffled, stratified folds. Grid search selects hyperparameters by mean validation macro-F1 and refits each family's best configuration on all development records. The search covers **23 configurations**, requiring **115 cross-validation fits and eight best-configuration refits** in full mode. Quick mode requires 69 cross-validation fits and refits on the search sample.

Families are ranked by mean macro-F1, with accuracy breaking ties. The top four enter threshold selection. Validation scores are unweighted. Fold standard deviations describe variation across folds, not confidence intervals; selecting configurations on these folds can make the best scores optimistic.


### Preprocessing within each training fold

Numeric features use median imputation with indicators for missingness observed during fitting. Categorical missing values receive `__missing__`, followed by one-hot encoding so nominal categories have no imposed order.

Categories appearing fewer than 20 times in a training fold share an infrequent category. LGA, district, qualified ward, and coordinate-grid fields have no additional category cap. Funder and installer each allow up to 300 encoded categories; other categorical fields allow up to 50. These caps include the infrequent category. Unseen values map to that category when it exists; otherwise, their indicators are all zero. Grouping limits feature count but can remove useful local detail.

KNN and logistic regression additionally standardize numeric inputs because distances and regularized coefficients depend on scale. Tree models use unscaled inputs. The encoded matrix is sparse to limit memory use; its width depends on the categories learned in each fit.


In [ ]:
invalidate_selection()

# Fit imputation and encoding within each training fold.

numeric_columns = X_dev_clean.select_dtypes(include="number").columns.tolist()
categorical_columns = X_dev_clean.columns.difference(numeric_columns).tolist()
geographic_columns = ["lga", "district_id", "ward_id", "geo_grid_005", "geo_grid_010"]
organisation_columns = ["funder", "installer"]
other_categorical_columns = [
    column
    for column in categorical_columns
    if column not in geographic_columns + organisation_columns
]


def make_categorical_pipeline(max_categories):
    return Pipeline(
        [
            ("impute", SimpleImputer(strategy="constant", fill_value="__missing__")),
            (
                "encode",
                OneHotEncoder(
                    handle_unknown="infrequent_if_exist",
                    min_frequency=20,
                    max_categories=max_categories,
                    sparse_output=True,
                ),
            ),
        ]
    )


def make_pipeline(model, scale_numeric=False):
    numeric_steps = [("impute", SimpleImputer(strategy="median", add_indicator=True))]

    if scale_numeric:
        numeric_steps.append(("scale", StandardScaler()))

    preprocess = ColumnTransformer(
        [
            ("numeric", Pipeline(numeric_steps), numeric_columns),
            ("geographic", make_categorical_pipeline(None), geographic_columns),
            ("organisation", make_categorical_pipeline(300), organisation_columns),
            ("categorical", make_categorical_pipeline(50), other_categorical_columns),
        ],
        sparse_threshold=1.0,
    )

    return Pipeline(
        [("preprocess", preprocess), ("model", model)], memory=pipeline_memory
    )


class ClassWeightedXGBoost(ClassifierMixin, BaseEstimator):
    """Compute training weights inside each fit, including each CV fold."""

    def __init__(self, estimator, class_weight=None):
        self.estimator = estimator
        self.class_weight = class_weight

    def fit(self, X, y):
        self.estimator_ = clone(self.estimator)
        weights = compute_sample_weight(self.class_weight, y)
        self.estimator_.fit(X, y, sample_weight=weights)
        self.classes_ = self.estimator_.classes_
        self.n_features_in_ = self.estimator_.n_features_in_
        return self

    def predict(self, X):
        check_is_fitted(self, "estimator_")
        return self.estimator_.predict(X)

    def predict_proba(self, X):
        check_is_fitted(self, "estimator_")
        return self.estimator_.predict_proba(X)


def predict_in_batches(model, frame, probabilities=False):
    """Limit memory use when predicting labels or class probabilities."""
    predict = model.predict_proba if probabilities else model.predict
    return np.concatenate(
        [
            predict(frame.iloc[start : start + BATCH_SIZE])
            for start in range(0, len(frame), BATCH_SIZE)
        ]
    )


In [ ]:
invalidate_selection()
# Quick mode uses a stratified development sample; full mode uses all rows.
if QUICK_RUN:
    X_fit, _, y_fit, _ = train_test_split(
        X_dev_clean,
        y_dev,
        train_size=min(9000, len(X_dev_clean) - 1),
        stratify=y_dev,
        random_state=SEED,
    )
else:
    X_fit, y_fit = X_dev_clean, y_dev
assert X_dev_clean.index.intersection(X_holdout_clean.index).empty
search_from_quick_run = QUICK_RUN

cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

folds = list(cv.split(X_fit, y_fit))
# Test repair thresholds on the validation portion of the first CV fold.
threshold_train_indices, threshold_validation_indices = folds[0]
X_threshold = X_fit.iloc[threshold_validation_indices]
y_threshold = y_fit.iloc[threshold_validation_indices]

scoring = {
    "accuracy": "accuracy",
    "macro_f1": "f1_macro",
    "repair_recall": make_scorer(
        recall_score,
        labels=[1],  # Class 1 means functional needs repair.
        average="macro",
        zero_division=0,
    ),
}

assert PRIMARY_METRIC in scoring

print("Rows used by every model:", len(X_fit))
print("Threshold-testing rows from the first CV fold:", len(X_threshold))
print("CV folds:", len(folds))
print("Selection metric:", PRIMARY_METRIC)


### Model choices and search scope

| Model | Searched settings | Fixed settings | Configurations |
|---|---|---|---:|
| Majority baseline | None | Most frequent training class | 1 |
| KNN | 5 or 11 neighbors | Distance weighting | 2 |
| Multinomial logistic regression | `C`: 0.3 or 1.0 | Balanced class weights | 2 |
| Decision tree | Depth: 20 or unlimited | Minimum leaf size 1; balanced class weights | 2 |
| Random forest | Feature fraction: 0.3 or 0.5; leaf size: 1 or 2 | 150 trees; unweighted training | 4 |
| Extra Trees | Feature fraction: 0.3 or 0.5; leaf size: 1 or 2 | 300 trees; unweighted training | 4 |
| Bagging trees | Sample fraction: 0.7 or 1.0; feature fraction: 0.5 or 0.75 | 300 bootstrap trees; base-tree leaf size 2; balanced class weights | 4 |
| XGBoost | Boosting rounds: 750 or 1,000; learning rate: 0.05 or 0.1 | Depth 10; unweighted training | 4 |

Fixed settings and candidate values retain each family's best configuration from a broader development-set search. The compact grid tests nearby alternatives while limiting computation. Earlier results informed these choices, so this comparison is part of model tuning rather than independent confirmation.

Logistic regression uses `lbfgs` with a 3,000-iteration limit; smaller `C` means stronger regularization. XGBoost uses `multi:softprob`, histogram-based tree construction, and fixed settings `min_child_weight=2`, `gamma=0.1`, `subsample=0.9`, `colsample_bytree=0.9`, `reg_alpha=0.1`, and `reg_lambda=0.5`. Its configured evaluation metric is multiclass log loss; cross-validation macro-F1 selects the configuration. No early stopping is applied.

### Training weights

Bagging base trees, logistic regression, and the standalone decision tree use inverse-frequency class weights:

$$w_k = \frac{n}{K n_k}, \qquad K=3,$$

where $n$ is the number of records in the classifier's training fit and $n_k$ is the count in class $k$. Weights are recalculated from the training labels within each fit. For all 59,400 labeled records, the corresponding weights are approximately **0.614**, **4.587**, and **0.868** for functional, repair, and non-functional respectively.

Random forest, Extra Trees, and XGBoost retain the unweighted settings selected by the broader comparison. KNN and the majority baseline also use unweighted class labels; KNN's distance weighting controls neighbor influence rather than class importance. The chosen weighting rules are preserved during threshold fitting and final refitting.


In [ ]:
invalidate_selection()

# Fix selected settings and compare a compact set of nearby alternatives.

experiments = {
    "Majority baseline": (
        make_pipeline(DummyClassifier(strategy="most_frequent", random_state=SEED)),
        {},
    ),
    "KNN": (
        make_pipeline(
            KNeighborsClassifier(algorithm="brute", weights="distance", n_jobs=2),
            scale_numeric=True,
        ),
        {"model__n_neighbors": [5, 11]},
    ),
    "Multinomial logistic regression": (
        make_pipeline(
            # lbfgs optimizes multinomial loss for the three-class task.
            LogisticRegression(
                solver="lbfgs",
                max_iter=3000,
                random_state=SEED,
                class_weight="balanced",
            ),
            scale_numeric=True,
        ),
        {"model__C": [0.3, 1.0]},
    ),
    "Decision tree": (
        make_pipeline(
            DecisionTreeClassifier(
                random_state=SEED, class_weight="balanced", min_samples_leaf=1
            )
        ),
        {"model__max_depth": [20, None]},
    ),
    "Random forest": (
        make_pipeline(
            RandomForestClassifier(
                n_estimators=150, random_state=SEED, n_jobs=2, class_weight=None
            )
        ),
        {
            "model__max_features": [0.3, 0.5],
            "model__min_samples_leaf": [1, 2],
        },
    ),
    "Extra Trees": (
        make_pipeline(
            ExtraTreesClassifier(
                n_estimators=300, random_state=SEED, n_jobs=2, class_weight=None
            )
        ),
        {
            "model__max_features": [0.3, 0.5],
            "model__min_samples_leaf": [1, 2],
        },
    ),
    "Bagging trees": (
        make_pipeline(
            BaggingClassifier(
                estimator=DecisionTreeClassifier(
                    random_state=SEED, class_weight="balanced", min_samples_leaf=2
                ),
                n_estimators=300,
                bootstrap=True,
                random_state=SEED,
                n_jobs=2,
            )
        ),
        {
            "model__max_samples": [0.7, 1.0],
            "model__max_features": [0.5, 0.75],
        },
    ),
    "XGBoost": (
        make_pipeline(
            ClassWeightedXGBoost(
                estimator=XGBClassifier(
                    objective="multi:softprob",
                    num_class=3,
                    eval_metric="mlogloss",
                    tree_method="hist",
                    # Fixed parameters selected by development-set cross-validation.
                    max_depth=10,
                    min_child_weight=2,
                    gamma=0.1,
                    subsample=0.9,
                    colsample_bytree=0.9,
                    reg_alpha=0.1,
                    reg_lambda=0.5,
                    random_state=SEED,
                    n_jobs=2,
                ),
                class_weight=None,
            )
        ),
        {
            # Compare boosting rounds and learning rate at the selected depth.
            "model__estimator__n_estimators": [750, 1000],
            "model__estimator__learning_rate": [0.05, 0.1],
        },
    ),
}

print("Models:", list(experiments))


In [ ]:
invalidate_selection()
searches = {}
rows = []

for name, (pipeline, parameters) in experiments.items():
    print(f"\nStarting: {name}", flush=True)
    start = perf_counter()
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=parameters,
        scoring=scoring,
        refit=PRIMARY_METRIC,
        cv=folds,
        n_jobs=1,
        error_score="raise",
        verbose=1,
    )

    # Share input arrays across threads to limit memory use.
    with parallel_backend("threading", n_jobs=2):
        search.fit(X_fit, y_fit)
    best_parameters = search.best_params_.copy()

    searches[name] = search
    best_index = search.best_index_
    scores = search.cv_results_
    row = {
        "model": name,
        "search_seconds": perf_counter() - start,
        "best_parameters": best_parameters,
    }
    for metric in scoring:
        row["cv_" + metric] = scores["mean_test_" + metric][best_index]
        row["sd_" + metric] = scores["std_test_" + metric][best_index]
    rows.append(row)

    # Retain fitted models for the leading four families and the baseline.
    keep_names = set(
        pd.DataFrame(rows)
        .sort_values(["cv_macro_f1", "cv_accuracy"], ascending=False, kind="stable")[
            "model"
        ]
        .head(4)
    ) | {"Majority baseline"}
    for discarded_name in list(searches):
        if discarded_name not in keep_names:
            del searches[discarded_name]

    filename = name.lower().replace(" ", "_") + "_search.csv"
    pd.DataFrame(search.cv_results_).to_csv(OUTPUT_DIR / filename, index=False)
    print("Best parameters:", best_parameters, flush=True)
    del search

results = (
    pd.DataFrame(rows)
    .sort_values(["cv_macro_f1", "cv_accuracy"], ascending=False, kind="stable")
    .reset_index(drop=True)
)
results.to_csv(OUTPUT_DIR / "model_comparison.csv", index=False)
print("Provisional CV leader:", results.loc[0, "model"])
search_ready = True


### Cross-validation results

The table presents each family's best configuration in descending order of mean cross-validation macro-F1. Accuracy and repair recall are proportions. `Accuracy SD` is the standard deviation of accuracy across the five folds.

These scores use the default argmax prediction rule. The following threshold comparison evaluates alternative repair decision rules for the four leading families.


In [ ]:
display(
    results[["model", "cv_accuracy", "cv_macro_f1", "cv_repair_recall", "sd_accuracy"]]
    .rename(
        columns={
            "model": "Model",
            "cv_accuracy": "Accuracy",
            "cv_macro_f1": "Macro-F1",
            "cv_repair_recall": "Repair recall",
            "sd_accuracy": "Accuracy SD",
        }
    )
    .round(4)
)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 6))

metrics = [
    ("cv_accuracy", "CV accuracy"),
    ("cv_macro_f1", "CV macro-F1"),
    ("cv_repair_recall", "CV repair-class recall"),
]

for ax, (metric, title) in zip(axes, metrics):
    sns.barplot(
        data=results,
        x=metric,
        y="model",
        order=results["model"].tolist(),
        color="steelblue",
        errorbar=None,
        ax=ax,
    )

    ax.set(title=title, xlabel="Score", ylabel="", xlim=(0, 1))

plt.tight_layout()
plt.show()


### Interpretation of the model comparison

**Bagging trees** leads with mean macro-F1 **0.7066**, accuracy **80.30%**, and repair recall **46.33%**. Its selected configuration uses **300 trees**, bootstrap samples with as many draws as training records (`max_samples=1.0`), feature fraction **0.5**, and minimum base-tree leaf size **2**. Macro-F1 standard deviation is **0.0027**; accuracy standard deviation is **0.15 percentage points**.

Extra Trees ranks second with macro-F1 **0.6973**, followed by random forest at **0.6968** and XGBoost at **0.6927**. Random forest has the highest accuracy (**81.15%**) but lower repair recall (**33.64%**) than bagging. The accuracy and macro-F1 rankings therefore differ.

Logistic regression has the highest repair recall (**71.45%**), alongside macro-F1 **0.6229** and accuracy **69.61%**. High repair recall alone does not establish stronger classification across all classes.

The majority baseline achieves macro-F1 **0.2346**, accuracy **54.31%**, and zero repair recall. Bagging improves accuracy by **25.99 percentage points** over this baseline. Its **0.0093** macro-F1 advantage over Extra Trees is descriptive; this comparison does not establish statistical significance.


### Select a model and repair threshold

The **top four families by mean cross-validation macro-F1** enter decision-rule selection. Each best configuration is fitted on the first fold's **42,768 training records** and predicts probabilities for its **10,692 validation records**, adding four fits. In quick mode, these sizes are 6,000 and 3,000 records.

Each model is evaluated using **argmax** and **91 repair thresholds from 0.05 to 0.95**. Argmax selects the class with the highest probability. The threshold rule predicts repair when its probability reaches the chosen threshold; otherwise, it selects the more probable of functional and non-functional.

Candidates are ranked by first-fold macro-F1, then accuracy, repair recall, and preference for argmax. Remaining ties retain shortlist order. These validation labels also informed hyperparameter selection, so the scores are tuning results rather than independent performance estimates.

**XGBoost with a repair threshold of 0.27 is selected**, achieving first-fold macro-F1 **0.7067**, compared with **0.6961** under argmax: a gain of **0.0106**. Its selected configuration uses depth **10**, **1,000 boosting rounds**, learning rate **0.1**, and unweighted training. Bagging's best threshold of **0.52** achieves macro-F1 **0.7048** on the same fold.

The chart shows XGBoost's threshold trade-offs; dotted lines show its argmax scores. Candidate scores are exported to `threshold_search.csv`, with each family's best rule in `threshold_model_comparison.csv`.

The selected threshold is applied to XGBoost already refitted on **all 53,460 development records**. The holdout evaluates this fixed model and rule without further threshold tuning.


In [ ]:
if not search_ready:
    raise RuntimeError("Complete hyperparameter search before threshold selection.")
invalidate_selection(invalidate_search=False)

# Tune decision rules using development predictions only; exclude holdout labels.


def predict_with_repair_threshold(probabilities, classes, threshold=None):
    classes = np.asarray(classes)
    if threshold is None:
        return classes[np.argmax(probabilities, axis=1)]
    repair_column = np.flatnonzero(classes == 1).item()
    other_columns = np.flatnonzero(classes != 1)
    predictions = classes[
        other_columns[np.argmax(probabilities[:, other_columns], axis=1)]
    ]
    predictions[probabilities[:, repair_column] >= threshold] = 1
    return predictions


class RepairThresholdClassifier(ClassifierMixin, BaseEstimator):
    """Apply a fixed repair threshold while supporting estimator refitting."""

    def __init__(self, estimator, repair_threshold=None):
        self.estimator = estimator
        self.repair_threshold = repair_threshold

    def fit(self, X, y):
        self.estimator_ = clone(self.estimator)
        self.estimator_.fit(X, y)
        self.classes_ = self.estimator_.classes_
        self.n_features_in_ = self.estimator_.n_features_in_
        return self

    def predict_proba(self, X):
        check_is_fitted(self, "estimator_")
        return self.estimator_.predict_proba(X)

    def predict(self, X):
        return predict_with_repair_threshold(
            self.predict_proba(X), self.classes_, self.repair_threshold
        )


# Results are ranked by macro-F1, with accuracy breaking ties.
threshold_model_names = results["model"].head(4).tolist()
print("Threshold shortlist:", threshold_model_names)
threshold_rows = []
repair_thresholds = [None] + np.round(np.linspace(0.05, 0.95, 91), 2).tolist()
for model_name in threshold_model_names:
    base_model = searches[model_name].best_estimator_
    threshold_classes = np.asarray(base_model.classes_)
    # Use first-fold validation predictions for decision-rule selection.
    # These labels also informed hyperparameter selection, so scores are tuning results.
    print(f"{model_name}: fitting the first CV fold for threshold testing", flush=True)
    fold_model = clone(base_model)
    with parallel_backend("threading", n_jobs=2):
        fold_model.fit(
            X_fit.iloc[threshold_train_indices], y_fit.iloc[threshold_train_indices]
        )
    assert np.array_equal(fold_model.classes_, threshold_classes)
    validation_probabilities = predict_in_batches(
        fold_model, X_threshold, probabilities=True
    )
    assert validation_probabilities.shape == (len(y_threshold), len(threshold_classes))
    assert np.isfinite(validation_probabilities).all()
    del fold_model

    for threshold in repair_thresholds:
        predictions = predict_with_repair_threshold(
            validation_probabilities, threshold_classes, threshold
        )
        threshold_rows.append(
            {
                "model": model_name,
                "rule": "default_argmax" if threshold is None else "repair_threshold",
                "repair_threshold": threshold,
                "macro_f1": f1_score(
                    y_threshold, predictions, average="macro", zero_division=0
                ),
                "accuracy": accuracy_score(y_threshold, predictions),
                "repair_precision": precision_score(
                    y_threshold,
                    predictions,
                    labels=[1],
                    average="macro",
                    zero_division=0,
                ),
                "repair_recall": recall_score(
                    y_threshold,
                    predictions,
                    labels=[1],
                    average="macro",
                    zero_division=0,
                ),
                "prefer_default": int(threshold is None),
            }
        )

threshold_results = pd.DataFrame(threshold_rows)
ranked_thresholds = threshold_results.sort_values(
    ["macro_f1", "accuracy", "repair_recall", "prefer_default"],
    ascending=False,
    kind="stable",
)
# Apply the same score ordering within models and across the shortlist.
threshold_model_summary = (
    ranked_thresholds.drop_duplicates("model").drop(columns="prefer_default").copy()
)
threshold_results = threshold_results.drop(columns="prefer_default")
default_by_model = threshold_results.loc[
    threshold_results["rule"] == "default_argmax"
].set_index("model")["macro_f1"]
threshold_model_summary["default_macro_f1"] = threshold_model_summary["model"].map(
    default_by_model
)
threshold_model_summary["macro_f1_gain"] = (
    threshold_model_summary["macro_f1"] - threshold_model_summary["default_macro_f1"]
)
best_rule = threshold_model_summary.iloc[0]
winner_name = best_rule["model"]
selected_repair_threshold = (
    None
    if best_rule["rule"] == "default_argmax"
    else float(best_rule["repair_threshold"])
)
threshold_results.to_csv(OUTPUT_DIR / "threshold_search.csv", index=False)
threshold_model_summary.to_csv(
    OUTPUT_DIR / "threshold_model_comparison.csv", index=False
)
display(threshold_model_summary.round(4))
print("Selected model after threshold tuning:", winner_name)
print("Selected prediction rule:", best_rule["rule"])
print("Selected repair threshold:", selected_repair_threshold)
print("First-fold threshold-testing macro-F1:", round(best_rule["macro_f1"], 4))


def plot_selected_thresholds(all_scores, selected_model, selected_threshold):
    """Plot the winning model, including a reference threshold if argmax wins."""
    model_scores = all_scores.loc[all_scores["model"] == selected_model]
    curves = model_scores.loc[model_scores["rule"] == "repair_threshold"]
    default_scores = model_scores.loc[model_scores["rule"] == "default_argmax"].iloc[0]
    # Argmax has no fixed repair threshold. Mark the best explicit alternative
    # for comparison without changing the selected prediction rule.
    if selected_threshold is None:
        best_explicit = curves.sort_values(
            ["macro_f1", "accuracy", "repair_recall"],
            ascending=False,
            kind="stable",
        ).iloc[0]
        marked_threshold = float(best_explicit["repair_threshold"])
        marker_label = (
            f"Best tested threshold: {marked_threshold:.2f} (argmax selected)"
        )
        rule_title = "selected rule: default argmax"
    else:
        marker_label = f"Selected threshold: {selected_threshold:.2f}"
        rule_title = f"selected repair threshold: {selected_threshold:.2f}"
        marked_threshold = selected_threshold

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    metric_groups = [("macro_f1", "accuracy"), ("repair_precision", "repair_recall")]
    for ax, metrics in zip(axes, metric_groups):
        for metric in metrics:
            (line,) = ax.plot(
                curves["repair_threshold"],
                curves[metric],
                label=metric.replace("_", " ").capitalize(),
            )
            ax.axhline(
                default_scores[metric], color=line.get_color(), linestyle=":", alpha=0.7
            )
        ax.axvline(marked_threshold, color="black", linestyle="--", label=marker_label)
        ax.set(xlabel="Repair probability threshold", ylabel="Score", ylim=(0, 1))
        ax.legend(fontsize=9)
    axes[0].set_title("Overall classification scores")
    axes[1].set_title("Repair precision and recall")
    fig.suptitle(
        f"{selected_model} — {rule_title}\nDotted lines: default argmax scores"
    )
    fig.tight_layout()
    return fig, axes


# Plot the selected model's threshold trade-offs.
fig, axes = plot_selected_thresholds(
    threshold_results, winner_name, selected_repair_threshold
)
plt.show()

# Reuse the search refit, which was trained on all model-search records.
base_winner = searches[winner_name].best_estimator_
winner = RepairThresholdClassifier(clone(base_winner), selected_repair_threshold)
winner.estimator_ = base_winner
winner.classes_ = base_winner.classes_
winner.n_features_in_ = base_winner.n_features_in_
del validation_probabilities, base_model, base_winner


## 7. Holdout evaluation

**XGBoost with repair threshold 0.27** achieves **81.18% accuracy** and **0.7144 macro-F1** on **5,940 holdout records**. The majority baseline achieves **54.31%** and **0.2346**, an accuracy difference of **26.87 percentage points**.

Repair detection remains the main weakness: precision is **46.71%**, recall **47.69%**, and F1 **0.4719**. The model correctly labels **206 of 432 repair cases**. Functional recall is **88.38%**; non-functional recall is **77.34%**.

The confusion matrices show counts and proportions within each actual class. Of the repair cases, **175** are classified as functional and **51** as non-functional. The latter remain flagged by the inspection policy despite their incorrect three-class label. Classification accuracy and inspection coverage therefore measure different outcomes.


In [ ]:
# Evaluate the frozen model and decision rule after development-only selection.

holdout_evaluated = False
evaluated_signature = None

if QUICK_RUN or not RUN_FINAL_EVALUATION:
    print("Holdout evaluation is not included in this run.")
else:
    if not search_ready or "winner" not in globals() or search_from_quick_run:
        raise RuntimeError("Complete full-data model and threshold selection first.")

    holdout_pred = predict_in_batches(winner, X_holdout_clean)

    comparisons = [(winner_name, holdout_pred)]
    if winner_name != "Majority baseline":
        baseline = searches["Majority baseline"].best_estimator_
        baseline_pred = predict_in_batches(baseline, X_holdout_clean)
        comparisons.append(("Majority baseline", baseline_pred))

    final_scores = pd.DataFrame(
        [
            {
                "model": name,
                "holdout_accuracy": accuracy_score(y_holdout, prediction),
                "holdout_macro_f1": f1_score(
                    y_holdout, prediction, average="macro", zero_division=0
                ),
            }
            for name, prediction in comparisons
        ]
    )
    display(final_scores)
    final_scores.to_csv(OUTPUT_DIR / "holdout_scores.csv", index=False)

    print(
        classification_report(
            y_holdout,
            holdout_pred,
            labels=[0, 1, 2],
            target_names=CLASS_NAMES,
            zero_division=0,
        )
    )

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    for ax, normalization, title, number_format in [
        (axes[0], None, "Holdout: counts", "d"),
        (axes[1], "true", "Holdout: proportions within actual class", ".1%"),
    ]:
        matrix = confusion_matrix(
            y_holdout, holdout_pred, labels=[0, 1, 2], normalize=normalization
        )

        sns.heatmap(
            matrix,
            annot=True,
            fmt=number_format,
            cmap="Blues",
            xticklabels=CLASS_NAMES,
            yticklabels=CLASS_NAMES,
            ax=ax,
        )

        ax.set(title=title, xlabel="Predicted", ylabel="Actual")
        ax.tick_params(axis="x", rotation=25)

    plt.tight_layout()
    plt.show()

    evaluated_signature = current_selection_signature()
    holdout_evaluated = True


## 8. Practical impact: prioritizing inspections

The model prioritizes waterpoints for inspection using the following rule:

- **Predicted functional:** skip inspection.
- **Predicted functional needs repair or non-functional:** select for inspection.

An **attention case** is a waterpoint whose actual label is functional needs repair or non-functional. Using the known holdout labels, we estimate how many sites would be inspected, how many attention cases would be reached, and how many would be missed.

A repair case predicted as non-functional still gets selected for inspection. This is why inspection coverage differs from correctly predicting all three classes.


In [ ]:
# Classes 1 (repair) and 2 (non-functional) count as attention cases.

if holdout_evaluated:
    require_current_holdout()
    actual_attention = y_holdout.to_numpy() != 0
    selected = holdout_pred != 0

    total_sites = len(y_holdout)
    sites_selected = int(selected.sum())
    attention_total = int(actual_attention.sum())
    attention_selected = int(np.sum(selected & actual_attention))
    attention_missed = int(np.sum(~selected & actual_attention))
    functional_selected = int(np.sum(selected & ~actual_attention))

    sites_selected_share = sites_selected / max(total_sites, 1)
    coverage = attention_selected / max(attention_total, 1)

    display(
        pd.DataFrame(
            {
                "Measure": [
                    "Sites selected for inspection",
                    "Attention cases selected",
                    "Attention cases missed",
                    "Functional sites selected",
                ],
                "Result": [
                    f"{sites_selected:,} of {total_sites:,} ({sites_selected_share:.2%})",
                    f"{attention_selected:,} of {attention_total:,} ({coverage:.2%})",
                    f"{attention_missed:,}",
                    f"{functional_selected:,}",
                ],
            }
        ).set_index("Measure")
    )


### Inspection workload and coverage

| Measure | Result | Meaning |
|---|---:|---|
| Sites selected for inspection | **2,456 of 5,940 (41.35%)** | The share of sites that would receive a visit. |
| Attention cases selected | **2,081 of 2,714 (76.68%)** | The share of pumps needing attention that would be reached. |
| Attention cases missed | **633** | Pumps needing attention that would be skipped. |
| Functional sites selected | **375** | Selected pumps that were actually functional. |

**Inspecting about 41% of sites would reach about 77% of attention cases in this holdout.** This suggests the model could help concentrate inspection visits on pumps likely to need attention.

However, almost a quarter of attention cases would be missed. These figures estimate inspection workload and coverage from recorded labels; a field evaluation would be needed to establish actual maintenance benefits.


## 9. Refit and create the submission

After holdout evaluation, the selected XGBoost configuration is fitted on **all 59,400 labeled waterpoints**. Unweighted training and the **0.27 repair threshold** are preserved. Holdout scores describe the development-trained model, not this final refit.

Predictions are converted to the exact competition labels and aligned with the submission template by `id`. Checks verify columns, row count, ID order, complete labels, and allowed values. The exported `outputs/submission.csv` contains **14,850 predictions**.

Prediction uses batches of 500 records to limit memory use. The decision rule receives no further tuning on holdout or competition data; competition labels are unavailable for local evaluation.


In [ ]:
# Refit the selected estimator on all labels while preserving its decision rule.
# Restore the exact competition class names before writing predictions.

if not holdout_evaluated:
    print("Competition predictions are not generated in this run.")
else:
    require_current_holdout()
    final_model = clone(winner)
    with parallel_backend("threading", n_jobs=2):
        final_model.fit(engineer_features(X), y)

    test_predictions = predict_in_batches(final_model, X_test_clean)

    predictions_by_id = pd.Series(test_predictions, index=test_values["id"]).map(
        number_to_label
    )

    # Map by ID rather than assuming the template has the same row order.
    submission = submission_format[["id"]].copy()
    submission["status_group"] = submission["id"].map(predictions_by_id)

    assert submission.columns.tolist() == ["id", "status_group"]
    assert len(submission) == len(test_values)
    assert submission["id"].equals(submission_format["id"])
    assert submission["status_group"].notna().all()
    assert set(submission["status_group"]).issubset(CLASS_NAMES)

    submission_path = OUTPUT_DIR / "submission.csv"
    submission.to_csv(submission_path, index=False)

    display(submission.head())
    print("Saved:", submission_path)


## 10. Limitations and practical use

- **Repair detection:** repair recall is 47.69%, leaving more than half of repair cases without the correct class label. Combining repair and non-functional predictions improves attention-case coverage but does not resolve this classification weakness.
- **Selection and evaluation:** hyperparameter and threshold selection share validation records, and earlier development results informed the compact grid. Holdout results have also been reviewed during development; confirmation on fresh data would strengthen the evidence.
- **Geographic and temporal transfer:** random splits can place nearby sites in both sets. Validation across regions or collection periods is needed to assess transfer to new settings.
- **Features and search scope:** dropping detailed fields, grouping rare categories, and fixing selected hyperparameters can exclude useful signal or stronger configurations. Predicted probabilities have not been separately assessed for calibration.
- **Operational objectives:** some predictors may require a site visit. Macro-F1 measures three-class classification, while inspection decisions depend on coverage, capacity, and locally justified costs.
- **Scope of evidence:** the labels describe recorded operating condition. The analysis does not establish future failure risk, water safety, maintenance benefits, or financial savings.


## 11. Overall conclusion

Bagging has the highest five-fold macro-F1 under argmax (**0.7066**), but the first-fold decision-rule comparison selects **XGBoost with repair threshold 0.27**. This model achieves holdout accuracy **81.18%** and macro-F1 **0.7144**.

Its inspection policy captures **76.68% of attention cases** while flagging **41.35% of sites**, with an inspection yield of **84.73%**. However, **633 attention cases remain unflagged**, and repair-class recall is **47.69%**. The results support further investigation of prediction-assisted inspection while exposing limits in repair detection and coverage.

Before operational use, evaluation should address geographic and temporal transfer, predictor availability, inspection capacity, and local costs. A field pilot would be needed to establish whether the policy improves maintenance outcomes.


## References

1. [DrivenData: Pump It Up dataset and target definitions](https://www.drivendata.org/competitions/7/pump-it-up-data-mining-the-water-table/page/25/).
2. [Scikit-learn: GridSearchCV and best-estimator refitting](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html).
3. [Scikit-learn: cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html).
4. [Scikit-learn: balanced sample weights](https://scikit-learn.org/stable/modules/generated/sklearn.utils.class_weight.compute_sample_weight.html).
5. [Scikit-learn: classification metrics](https://scikit-learn.org/stable/modules/model_evaluation.html#classification-metrics).
6. [Scikit-learn: one-hot encoding and infrequent categories](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html).
7. [XGBoost: classifier API](https://xgboost.readthedocs.io/en/stable/python/python_api.html).
8. [Scikit-learn: decision-threshold selection](https://scikit-learn.org/stable/modules/classification_threshold.html).
